# EPIC Clarity Visit Occurrence Hydration

This notebook hydrates the OMOP VISIT_OCCURRENCE table from EPIC Clarity patient encounters.

## Source Tables
- `_exponent._bronze_epic_clarity.pat_enc` - Patient encounters (main source)
- `_exponent._bronze_epic_clarity.pat_enc_hsp` - Hospital encounter details

## OMOP Fields Populated
- visit_occurrence_id (surrogate key from mapping)
- person_id (linked from person mapping)
- visit_concept_id (mapped from encounter type)
- visit_start_date / visit_start_datetime
- visit_end_date / visit_end_datetime
- visit_type_concept_id (32817 = EHR)
- provider_id (attending provider)
- care_site_id (department)
- visit_source_value

In [ ]:
source = 'epic_clarity'

In [ ]:
silver_visit_occurrence_df = spark.sql("""
WITH visit_with_duplicates AS (
  SELECT
    p.person_id,
    0 AS visit_concept_id,
    CAST(pe.CONTACT_DATE AS DATE) AS visit_start_date,
    CAST(pe.CONTACT_DATE AS TIMESTAMP) AS visit_start_datetime,
    CAST(COALESCE(peh.HOSP_DISCH_TIME, pe.CONTACT_DATE) AS DATE) AS visit_end_date,
    CAST(COALESCE(peh.HOSP_DISCH_TIME, pe.CONTACT_DATE) AS TIMESTAMP) AS visit_end_datetime,
    32817 AS visit_type_concept_id,
    NULL AS provider_id,
    NULL AS care_site_id,
    CONCAT_WS(CHR(31), 'epic_clarity', 'PAT_ENC', 'PAT_ENC_CSN_ID', CAST(pe.PAT_ENC_CSN_ID AS STRING)) AS visit_source_value,
    0 AS visit_source_concept_id,
    0 AS admitted_from_concept_id,
    NULL AS admitted_from_source_value,
    0 AS discharged_to_concept_id,
    NULL AS discharged_to_source_value,
    NULL AS preceding_visit_occurrence_id,
    'epic_clarity' AS source_system,
    CURRENT_TIMESTAMP() AS last_mod_tsp,
    ROW_NUMBER() OVER (PARTITION BY CONCAT_WS(CHR(31), 'epic_clarity', 'PAT_ENC', 'PAT_ENC_CSN_ID', CAST(pe.PAT_ENC_CSN_ID AS STRING)) ORDER BY pe.CONTACT_DATE DESC) as rn
  FROM _exponent._bronze_epic_clarity.pat_enc pe
  INNER JOIN _exponent.omop_mapping.source_to_person p
    ON CONCAT_WS(CHR(31), 'epic_clarity', 'PATIENT', 'PAT_ID', CAST(pe.PAT_ID AS STRING)) = p.person_source_value
    AND p.active_flag = TRUE
  LEFT JOIN _exponent._bronze_epic_clarity.pat_enc_hsp peh
    ON pe.PAT_ENC_CSN_ID = peh.PAT_ENC_CSN_ID
  WHERE pe.PAT_ENC_CSN_ID IS NOT NULL
    AND pe.CONTACT_DATE IS NOT NULL
)
SELECT
  person_id,
  visit_concept_id,
  visit_start_date,
  visit_start_datetime,
  visit_end_date,
  visit_end_datetime,
  visit_type_concept_id,
  provider_id,
  care_site_id,
  visit_source_value,
  visit_source_concept_id,
  admitted_from_concept_id,
  admitted_from_source_value,
  discharged_to_concept_id,
  discharged_to_source_value,
  preceding_visit_occurrence_id,
  source_system,
  last_mod_tsp
FROM visit_with_duplicates
WHERE rn = 1
""")

# display(silver_visit_occurrence_df)
silver_visit_occurrence_df.createOrReplaceTempView("silver_visit_occurrence")

In [ ]:
%sql
MERGE INTO _exponent.omop_silver.visit_occurrence AS target
USING silver_visit_occurrence AS source
ON target.visit_source_value = source.visit_source_value

WHEN MATCHED AND NOT (
    target.person_id <=> source.person_id
    AND target.visit_start_date <=> source.visit_start_date
    AND target.visit_end_date <=> source.visit_end_date
)
THEN UPDATE SET
    target.person_id = source.person_id,
    target.visit_concept_id = source.visit_concept_id,
    target.visit_start_date = source.visit_start_date,
    target.visit_start_datetime = source.visit_start_datetime,
    target.visit_end_date = source.visit_end_date,
    target.visit_end_datetime = source.visit_end_datetime,
    target.visit_type_concept_id = source.visit_type_concept_id,
    target.provider_id = source.provider_id,
    target.care_site_id = source.care_site_id,
    target.visit_source_concept_id = source.visit_source_concept_id,
    target.admitted_from_concept_id = source.admitted_from_concept_id,
    target.admitted_from_source_value = source.admitted_from_source_value,
    target.discharged_to_concept_id = source.discharged_to_concept_id,
    target.discharged_to_source_value = source.discharged_to_source_value,
    target.preceding_visit_occurrence_id = source.preceding_visit_occurrence_id,
    target.last_mod_tsp = source.last_mod_tsp

WHEN NOT MATCHED THEN INSERT (
    person_id,
    visit_concept_id,
    visit_start_date,
    visit_start_datetime,
    visit_end_date,
    visit_end_datetime,
    visit_type_concept_id,
    provider_id,
    care_site_id,
    visit_source_value,
    visit_source_concept_id,
    admitted_from_concept_id,
    admitted_from_source_value,
    discharged_to_concept_id,
    discharged_to_source_value,
    preceding_visit_occurrence_id,
    source_system,
    last_mod_tsp
)
VALUES (
    source.person_id,
    source.visit_concept_id,
    source.visit_start_date,
    source.visit_start_datetime,
    source.visit_end_date,
    source.visit_end_datetime,
    source.visit_type_concept_id,
    source.provider_id,
    source.care_site_id,
    source.visit_source_value,
    source.visit_source_concept_id,
    source.admitted_from_concept_id,
    source.admitted_from_source_value,
    source.discharged_to_concept_id,
    source.discharged_to_source_value,
    source.preceding_visit_occurrence_id,
    source.source_system,
    source.last_mod_tsp
)

In [ ]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_visit_occurrence (
    visit_occurrence_source_value,
    active_flag,
    created_tsp
)
SELECT
    visit_source_value,
    TRUE AS active_flag,
    CURRENT_TIMESTAMP() AS created_tsp
FROM _exponent.omop_silver.visit_occurrence
WHERE visit_source_value IS NOT NULL
  AND source_system = 'epic_clarity'
  AND visit_source_value NOT IN (
    SELECT visit_occurrence_source_value
    FROM _exponent.omop_mapping.source_to_visit_occurrence
    WHERE active_flag = TRUE
  )

In [ ]:
gold_visit_occurrence_df = spark.sql("""
WITH gold_with_duplicates AS (
  SELECT
    m.visit_occurrence_id,
    s.person_id,
    s.visit_concept_id,
    s.visit_start_date,
    s.visit_start_datetime,
    s.visit_end_date,
    s.visit_end_datetime,
    s.visit_type_concept_id,
    s.provider_id,
    s.care_site_id,
    s.visit_source_concept_id,
    s.admitted_from_concept_id,
    s.admitted_from_source_value,
    s.discharged_to_concept_id,
    s.discharged_to_source_value,
    s.preceding_visit_occurrence_id,
    ROW_NUMBER() OVER (PARTITION BY m.visit_occurrence_id ORDER BY s.last_mod_tsp DESC) as rn
  FROM _exponent.omop_silver.visit_occurrence s
  INNER JOIN _exponent.omop_mapping.source_to_visit_occurrence m
    ON s.visit_source_value = m.visit_occurrence_source_value
    AND m.active_flag = TRUE
  WHERE s.source_system = 'epic_clarity'
)
SELECT
  visit_occurrence_id,
  person_id,
  visit_concept_id,
  visit_start_date,
  visit_start_datetime,
  visit_end_date,
  visit_end_datetime,
  visit_type_concept_id,
  provider_id,
  care_site_id,
  visit_source_concept_id,
  admitted_from_concept_id,
  admitted_from_source_value,
  discharged_to_concept_id,
  discharged_to_source_value,
  preceding_visit_occurrence_id
FROM gold_with_duplicates
WHERE rn = 1
""")

# display(gold_visit_occurrence_df)
gold_visit_occurrence_df.createOrReplaceTempView("gold_visit_occurrence")

In [ ]:
%sql
MERGE INTO _exponent.omop.visit_occurrence AS target
USING gold_visit_occurrence AS source
ON target.visit_occurrence_id = source.visit_occurrence_id

WHEN MATCHED AND NOT (
    target.person_id <=> source.person_id
    AND target.visit_concept_id <=> source.visit_concept_id
    AND target.visit_start_date <=> source.visit_start_date
    AND target.visit_end_date <=> source.visit_end_date
)
THEN UPDATE SET
    target.person_id = source.person_id,
    target.visit_concept_id = source.visit_concept_id,
    target.visit_start_date = source.visit_start_date,
    target.visit_start_datetime = source.visit_start_datetime,
    target.visit_end_date = source.visit_end_date,
    target.visit_end_datetime = source.visit_end_datetime,
    target.visit_type_concept_id = source.visit_type_concept_id,
    target.provider_id = source.provider_id,
    target.care_site_id = source.care_site_id,
    target.visit_source_concept_id = source.visit_source_concept_id,
    target.admitted_from_concept_id = source.admitted_from_concept_id,
    target.admitted_from_source_value = source.admitted_from_source_value,
    target.discharged_to_concept_id = source.discharged_to_concept_id,
    target.discharged_to_source_value = source.discharged_to_source_value,
    target.preceding_visit_occurrence_id = source.preceding_visit_occurrence_id

WHEN NOT MATCHED THEN INSERT (
    visit_occurrence_id,
    person_id,
    visit_concept_id,
    visit_start_date,
    visit_start_datetime,
    visit_end_date,
    visit_end_datetime,
    visit_type_concept_id,
    provider_id,
    care_site_id,
    visit_source_concept_id,
    admitted_from_concept_id,
    admitted_from_source_value,
    discharged_to_concept_id,
    discharged_to_source_value,
    preceding_visit_occurrence_id
)
VALUES (
    source.visit_occurrence_id,
    source.person_id,
    source.visit_concept_id,
    source.visit_start_date,
    source.visit_start_datetime,
    source.visit_end_date,
    source.visit_end_datetime,
    source.visit_type_concept_id,
    source.provider_id,
    source.care_site_id,
    source.visit_source_concept_id,
    source.admitted_from_concept_id,
    source.admitted_from_source_value,
    source.discharged_to_concept_id,
    source.discharged_to_source_value,
    source.preceding_visit_occurrence_id
)